<details>
<summary><b>Info</b></summary>

**Last Execution:** 2026-07-27

| Package | Version |
|---------|---------|
| **nnsight** | **0.8** |
| Python | 3.12.13 |
| torch | 2.13.0+cu126 |
| transformers | 5.15.0 |

</details>


# Loading a Model

Before you can trace anything you need a model. nnsight never runs a bare
`torch.nn.Module` directly — it wraps one in an **Envoy** tree so every submodule
becomes observable. This page is about getting to that wrapped model: constructing
one from a HuggingFace repo id, understanding the `meta` build and when weights
actually load, wrapping a model you already have in memory, and choosing where the
weights live.

The workhorse wrapper is `TransformersModel`, which loads any HuggingFace
`transformers` checkpoint through a `pipeline` so tokenization and generation come
for free. For an arbitrary PyTorch module there is the thin `NNsight` wrapper.

In [1]:
from nnsight.modeling.transformers import TransformersModel
from nnsight import NNsight
import torch

/home/localjadenfk/miniconda3/envs/ndif2/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Constructing from a Repo ID

The first argument is a HuggingFace repo id. Any keyword you would pass to
`transformers` loading — `device_map`, `dtype` (the `torch_dtype` alias also
works), `revision`, `attn_implementation`, and so on — is forwarded through to the
underlying load, alongside a few nnsight-specific options like `dispatch` and
`rename`.

In [2]:
model = TransformersModel("openai-community/gpt2")

# The tokenizer the model will actually use is exposed as an attribute.
print(model.tokenizer)

GPT2Tokenizer(name_or_path='openai-community/gpt2', vocab_size=50257, model_max_length=1024, padding_side='left', truncation_side='right', special_tokens={'bos_token': '<|endoftext|>', 'eos_token': '<|endoftext|>', 'unk_token': '<|endoftext|>', 'pad_token': '<|endoftext|>'}, added_tokens_decoder={
	50256: AddedToken("<|endoftext|>", rstrip=False, lstrip=False, single_word=False, normalized=True, special=True),
})


That single line is enough to trace against. Because the model was built lazily
(see the next section), the weights load automatically the first time you run it.

In [3]:
with model.trace("The Eiffel Tower is in the city of"):
    logits = model.output.logits.save()

token = logits[0, -1].argmax(dim=-1)
print(f"Prediction: {model.tokenizer.decode(token)!r}")

Prediction: ' Paris'


## The Meta Model and Dispatching

By default `TransformersModel(repo_id)` does **not** download or allocate any
weights. It reads only the model's config and builds the full architecture on the
[`meta` device](https://pytorch.org/docs/stable/meta.html) — every module and
parameter exists with the right shape and dtype, but backed by no storage. This is
cheap and instant, and it is all nnsight needs to let you write interventions
against module paths: the tree is fully navigable before a single byte of weights
is loaded.

The `dispatched` flag tells you whether real weights are in memory yet.

In [4]:
meta_model = TransformersModel("openai-community/gpt2")

print("dispatched:", meta_model.dispatched)
print("weight device:", next(meta_model._module.parameters()).device)

dispatched: False
weight device: meta


Because the architecture is fully present, you can already inspect it and even
`scan` for activation shapes — that runs the forward under fake tensors, so it
needs no weights and never dispatches.

In [5]:
with meta_model.scan("Hello world"):
    shape = meta_model.transformer.h[0].output[0].shape.save()

print("block 0 output shape:", tuple(shape))
print("still on meta, dispatched:", meta_model.dispatched)

block 0 output shape: (2, 768)
still on meta, dispatched: False


**Dispatching** is the step that loads the real weights and swaps them into the
existing tree (in place, so every module path you already referenced stays valid).
It happens automatically on the first real trace or generation:

In [6]:
with meta_model.trace("Hello world"):
    pass

print("after first trace, dispatched:", meta_model.dispatched)
print("weight device:", next(meta_model._module.parameters()).device)

after first trace, dispatched: True
weight device: cuda:0


If you would rather pay that cost up front — for instance to fail fast on an OOM,
or to warm the model before timing anything — pass `dispatch=True` to load eagerly
at construction. The meta phase is skipped entirely.

In [7]:
eager_model = TransformersModel("openai-community/gpt2", dispatch=True)

print("dispatched immediately:", eager_model.dispatched)

dispatched immediately: True


<details class="admonition note">
<summary>Why a meta build at all?</summary>

Building on `meta` makes construction instant and memory-free, which matters most
for **remote** execution: your client builds the weightless skeleton so it knows
the module paths, while the real weights only ever live on the server. Locally it
means you can hold a `TransformersModel` for a checkpoint you have not decided to
load yet, and only pay for the weights when you actually trace.

</details>

## Wrapping an Already-Loaded Model

You do not have to hand nnsight a repo id. If you already have a model object in
memory, pass the object itself and nnsight wraps it as-is — no meta phase, already
dispatched.

For an arbitrary `torch.nn.Module`, use `NNsight`. It builds a root Envoy
mirroring the module tree, so children are reachable by index or attribute exactly
as in PyTorch.

In [8]:
net = torch.nn.Sequential(
    torch.nn.Linear(5, 10),
    torch.nn.Linear(10, 2),
)
wrapped = NNsight(net)

with wrapped.trace(torch.rand(1, 5)):
    hidden = wrapped[0].output.save()

print("hidden shape:", tuple(hidden.shape))

hidden shape: (1, 10)


For a HuggingFace model you loaded yourself, pass the loaded module to
`TransformersModel` instead of a repo id. It infers the pipeline task from the
model's architecture and reuses your instance directly — handy when you have
already customized the model (quantization, an adapter, edited weights) and want
to trace *that* object.

In [9]:
from transformers import AutoModelForCausalLM

hf_model = AutoModelForCausalLM.from_pretrained("openai-community/gpt2")
model_from_object = TransformersModel(hf_model)

print("inferred task:", model_from_object.task)
print("dispatched:", model_from_object.dispatched)

with model_from_object.trace("Hello"):
    logits = model_from_object.output.logits.save()

print("logits shape:", tuple(logits.shape))

inferred task: text-generation
dispatched: True
logits shape: (1, 1, 50257)


<details class="admonition note">
<summary>Task inference</summary>

The `pipeline` factory can infer a task from a repo-id *string*, but not from a
bare module instance. nnsight infers it from the architecture — a generative model
(`can_generate()`) becomes `text-generation`, otherwise the class-name suffix
decides (`*ForMaskedLM` → `fill-mask`, and so on). If it can't be inferred, pass
`task=...` explicitly, e.g. `TransformersModel(hf_model, task="text-generation")`.

</details>

## Device Placement

Where the weights land is decided at dispatch, by the same knobs `transformers`
uses. With a GPU available, `transformers` places the model on it by **default**,
so a plain `TransformersModel(repo_id)` runs on the GPU here without asking. To
choose explicitly:

- **`device="cpu"` / `device="cuda"` / `device=0`** — put the whole model on one
  device. This is how you pin to the CPU when a GPU would otherwise be picked.
- **`device_map="cuda"`** — the single GPU, via accelerate.
- **`device_map="auto"`** — let accelerate place and, if needed, shard the model
  across every available device (GPUs first, spilling to CPU/disk for a model too
  big to fit). This is what you want for large checkpoints.

`device` and `device_map` are HuggingFace loading options, so they only take
effect once the model dispatches.

In [10]:
# Force the model onto the CPU, overriding the GPU-by-default behavior.
cpu_model = TransformersModel("openai-community/gpt2", device="cpu", dispatch=True)
print("device=cpu       ->", next(cpu_model._module.parameters()).device)

# Pin to the single GPU.
gpu_model = TransformersModel("openai-community/gpt2", device="cuda", dispatch=True)
print("device=cuda      ->", next(gpu_model._module.parameters()).device)

# Let accelerate place (and shard, for big models) across available devices.
auto_model = TransformersModel("openai-community/gpt2", device_map="auto", dispatch=True)
print("device_map=auto  ->", next(auto_model._module.parameters()).device)

device=cpu       -> cpu


device=cuda      -> cuda:0


device_map=auto  -> cuda:0


Once loaded, the model is an ordinary wrapped module: from here every other
feature page — module access, batching, generation, editing — applies unchanged,
regardless of how you constructed it or where it lives.